# Run Airflow Operator Service

This notebook checks if a Google Cloud Run Service and its corresponding GCS bucket exist (creating them if necessary), then sends an execution request to the Cloud Run service.

In [ ]:
# Parameter Cell
PROJECT_ID = "__PLACEHOLDER_PROJECT_ID__"
LOCATION = "us-central1"
SERVICE_NAME = "airflow-operator-service"
SERVICE_ACCOUNT = "dataform-compiler@__PLACEHOLDER_PROJECT_ID__.iam.gserviceaccount.com"

In [ ]:
import time
import json
import urllib.request
import urllib.error
import google.auth
import google.auth.transport.requests

# --- Helper Functions ---

def get_access_token():
    """Obtains an active Google Cloud API access token."""
    try:
        credentials, _ = google.auth.default(
            scopes=['https://www.googleapis.com/auth/cloud-platform']
        )
        auth_req = google.auth.transport.requests.Request()
        credentials.refresh(auth_req)
        return credentials.token
    except Exception as e:
        print(f"Failed to obtain access token: {e}")
        raise e

def make_request(url, headers, method="GET", body=None):
    """Performs an HTTP request using urllib and returns the JSON response."""
    data = json.dumps(body).encode("utf-8") if body is not None else None
    req = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req) as response:
            res_body = response.read().decode("utf-8")
            return json.loads(res_body) if res_body else {}
    except urllib.error.HTTPError as e:
        err_msg = e.read().decode('utf-8')
        raise urllib.error.HTTPError(e.url, e.code, f"{e.msg}: {err_msg}", e.hdrs, e.fp)

def get_project_number(project_id, headers):
    """Retrieves numeric project number from metadata server or REST API."""
    # 1. Try metadata server
    try:
        req = urllib.request.Request(
            "http://metadata.google.internal/computeMetadata/v1/project/numeric-project-id",
            headers={"Metadata-Flavor": "Google"},
            timeout=2
        )
        with urllib.request.urlopen(req) as response:
            return response.read().decode("utf-8").strip()
    except Exception:
        pass
        
    # 2. Try Cloud Resource Manager API
    try:
        url = f"https://cloudresourcemanager.googleapis.com/v1/projects/{project_id}"
        res = make_request(url, headers, method="GET")
        return res.get("projectNumber")
    except Exception as e:
        print(f"Could not retrieve project number via API: {e}")
        
    # 3. Fallback default
    if project_id == "__PLACEHOLDER_PROJECT_ID__":
        return "1031557682594"
    return None

def ensure_bucket_exists(project_id, bucket_name, location, headers):
    """Checks if GCS bucket exists, and creates it if missing."""
    bucket_url = f"https://storage.googleapis.com/storage/v1/b/{bucket_name}"
    try:
        make_request(bucket_url, headers, method="GET")
        print(f"GCS Bucket {bucket_name} already exists.")
    except urllib.error.HTTPError as e:
        if e.code == 404:
            print(f"GCS Bucket {bucket_name} does not exist. Creating it...")
            create_url = f"https://storage.googleapis.com/storage/v1/b?project={project_id}"
            body = {"name": bucket_name, "location": location}
            make_request(create_url, headers, method="POST", body=body)
            print(f"GCS Bucket {bucket_name} created successfully.")
        else:
            raise e

def ensure_service_exists(project_id, location, service_name, project_number, bucket_name, service_account, headers):
    """Checks if Cloud Run service exists, and creates it if missing. Returns service URI."""
    service_url = f"https://{location}-run.googleapis.com/v2/projects/{project_id}/locations/{location}/services/{service_name}"
    try:
        service = make_request(service_url, headers, method="GET")
        print(f"Cloud Run Service {service_name} already exists.")
        return service.get("uri")
    except urllib.error.HTTPError as e:
        if e.code == 404:
            print(f"Cloud Run Service {service_name} does not exist. Creating it...")
            create_url = f"https://{location}-run.googleapis.com/v2/projects/{project_id}/locations/{location}/services?serviceId={service_name}"
            service_body = {
                "template": {
                    "timeout": "600s",
                    "containers": [{
                        "name": "runner",
                        "image": f"us-central1-docker.pkg.dev/{project_id}/composer-lite-images/csr-lite-runner-service:latest",
                        "env": [
                            {"name": "SERVERLESS_PROJECT_NUMBER", "value": project_number},
                            {"name": "SERVERLESS_BUCKET_NAME", "value": bucket_name}
                        ],
                        "resources": {
                            "limits": {"cpu": "1000m", "memory": "2Gi"}
                        },
                        "ports": [{
                            "containerPort": 8080
                        }]
                    }],
                    "serviceAccount": service_account
                }
            }
            op = make_request(create_url, headers, method="POST", body=service_body)
            poll_operation(location, op["name"], headers, "Service creation")
            print(f"Cloud Run Service {service_name} created successfully.")
            # Fetch the service details to obtain the URI
            service = make_request(service_url, headers, method="GET")
            return service.get("uri")
        else:
            raise e

def poll_operation(location, operation_name, headers, task_name="Operation"):
    """Polls a Cloud Run Long Running Operation (LRO) until done."""
    poll_url = f"https://{location}-run.googleapis.com/v2/{operation_name}"
    print(f"Waiting for {task_name}...")
    while True:
        op = make_request(poll_url, headers, method="GET")
        if op.get("done"):
            if "error" in op:
                raise RuntimeError(f"{task_name} failed: {op['error']}")
            return op
        time.sleep(2)

def get_id_token(audience):
    """Obtains an active Google Cloud API ID token for the specified audience."""
    import google.oauth2.id_token
    import google.auth.transport.requests
    try:
        credentials, _ = google.auth.default()
        auth_req = google.auth.transport.requests.Request()
        credentials.refresh(auth_req)
        return google.oauth2.id_token.fetch_id_token(auth_req, audience)
    except Exception as e:
        print(f"Standard google.auth ID token retrieval failed: {e}. Trying metadata server...")
        try:
            req = urllib.request.Request(
                f"http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/identity?audience={audience}",
                headers={"Metadata-Flavor": "Google"},
                timeout=2
            )
            with urllib.request.urlopen(req) as response:
                return response.read().decode("utf-8").strip()
        except Exception as e2:
            print(f"Metadata server ID token retrieval failed: {e2}")
            raise e2

def trigger_service(service_url, project_number, bucket_name):
    """Sends a synchronous execution request to the Cloud Run service and returns the response."""
    # Obtain OIDC ID token for the service URL
    id_token = get_id_token(service_url)
    headers = {
        "Authorization": f"Bearer {id_token}",
        "Content-Type": "application/json"
    }
    
    run_url = f"{service_url}/run"
    print(f"Triggering task execution on Cloud Run Service: {run_url}")
    
    body = {
        "dag_id": "my-serverless-dag",
        "run_id": "my-serverless-run",
        "task_id": "my-serverless-task",
        "try_number": 1,
        "dag_file": "__PLACEHOLDER_SERVERLESS_DAG_FILE_B64__"
    }
    
    return make_request(run_url, headers, method="POST", body=body)

# --- Main Flow ---

def main():
    global PROJECT_ID, SERVICE_ACCOUNT
    
    # 1. Setup Auth
    access_token = get_access_token()
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    # 2. Auto-detect Project ID if needed
    if not PROJECT_ID or PROJECT_ID == "__PLACEHOLDER_PROJECT_ID__":
        try:
            _, auto_project_id = google.auth.default()
            if auto_project_id:
                PROJECT_ID = auto_project_id
        except Exception as e:
            print(f"Could not auto-detect project ID: {e}. Using default: {PROJECT_ID}")

    # Resolve service account
    if not SERVICE_ACCOUNT:
        SERVICE_ACCOUNT = f"dataform-compiler@{PROJECT_ID}.iam.gserviceaccount.com"
    else:
        if "@__PLACEHOLDER_PROJECT_ID__.iam.gserviceaccount.com" in SERVICE_ACCOUNT:
            SERVICE_ACCOUNT = SERVICE_ACCOUNT.replace("__PLACEHOLDER_PROJECT_ID__", PROJECT_ID)

    print(f"Target Project: {PROJECT_ID}")
    print(f"Target Location: {LOCATION}")
    print(f"Target Service Name: {SERVICE_NAME}")
    print(f"Target Service Account: {SERVICE_ACCOUNT}")

    # 3. Retrieve numeric project number
    project_number = get_project_number(PROJECT_ID, headers)
    if not project_number:
        raise RuntimeError("Failed to retrieve project number for setting up service environment.")
    print(f"Resolved Project Number: {project_number}")

    # 4. Check/Create Bucket
    bucket_name = f"csr-lite-bucket-{project_number}"
    ensure_bucket_exists(PROJECT_ID, bucket_name, LOCATION, headers)

    # 5. Check/Create Cloud Run Service
    service_uri = ensure_service_exists(PROJECT_ID, LOCATION, SERVICE_NAME, project_number, bucket_name, SERVICE_ACCOUNT, headers)
    if not service_uri:
        raise RuntimeError(f"Failed to retrieve service URI for service: {SERVICE_NAME}")
    print(f"Service URI resolved: {service_uri}")

    # 6. Trigger Service Execution and wait for response
    execution = trigger_service(service_uri, project_number, bucket_name)
    print(f"Service Execution Response: {execution}")
    
    state = execution.get("state")
    error = execution.get("error")
    
    print(f"Task result state: {state}")
    if error:
        print(f"Task error details: {error}")
    
    if state == "failed" or error:
        raise RuntimeError(f"Cloud Run service task failed with state '{state}'.")
    
    print("Cloud Run service task completed successfully!")

# Run the flow
main()